In [1]:
import json
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path

import yaml
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
CONFIG_PATH = "/content/drive/MyDrive/Deepfake_Preprocessed/config.yaml"

def load_config(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    print(f"config 로드 완료: {path}")
    return cfg

CFG = load_config(CONFIG_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config 로드 완료: /content/drive/MyDrive/Deepfake_Preprocessed/config.yaml
Device: cuda


# Dataset

In [4]:
_image_size = CFG["input"]["image_size"]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((_image_size, _image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((_image_size, _image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [5]:
class DeepfakeDataset(Dataset):
    def __init__(self, base_dir, df, transform=None, max_frames=64):
        self.base_dir = Path(base_dir)
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.max_frames = max_frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frames_dir = self.base_dir / row["video_id"] / "frames"
        pngs = sorted(frames_dir.glob("*.png"))[: self.max_frames]

        if not pngs:
            raise RuntimeError(f"프레임 없음: {row['video_id']}")

        imgs = []
        for png in pngs:
            img = cv2.imread(str(png))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            imgs.append(img)

        # 부족한 프레임 마지막으로 패딩
        while len(imgs) < self.max_frames:
            imgs.append(imgs[-1].copy())

        if self.transform:
            seed = np.random.randint(0, 2**31)
            np.random.seed(seed)
            torch.manual_seed(seed)
            img_tensor = torch.stack([self.transform(img) for img in imgs])
        else:
            img_seq = np.stack(imgs, axis=0).transpose(0, 3, 1, 2)
            img_tensor = torch.from_numpy(img_seq).float() / 255.0

        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return img_tensor, label

# Model

In [6]:
class HighPassPreprocess(nn.Module):
    def __init__(self, strength=0.15):
        super().__init__()
        kernel = torch.tensor(
            [[0.0, -1.0, 0.0],
             [-1.0,  4.0, -1.0],
             [0.0, -1.0, 0.0]],
            dtype=torch.float32,
        )
        kernel = kernel.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.register_buffer("kernel", kernel)
        self.strength = strength

    def forward(self, x):
        hp = F.conv2d(x, self.kernel, padding=1, groups=3)
        return x + self.strength * hp

In [7]:
class ConvNeXtArtifactDetector(nn.Module):
    def __init__(self, model_name="convnext_tiny", pretrained=True, dropout=0.2, use_highpass=True):
        super().__init__()
        self.use_highpass = use_highpass
        self.pre = HighPassPreprocess(strength=0.15) if use_highpass else nn.Identity()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0, # backbone을 feature 추출기로만 사용하기 [num_classes=0이면 timm이 classifier 안붙임]
            global_pool="avg"
        )

        feat_dim = self.backbone.num_features
        self.head = nn.Sequential( # Head 직접 만들어 붙임
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 1)
        )

    def forward(self, x):
      # x : (B, T, C, H, W) | T: 영상에서 추출한 프레임 수
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        x = self.pre(x)

        feat = self.backbone(x)
        feat = feat.view(B, T, -1).mean(dim=1)

        logit = self.head(feat).squeeze(1)
        return logit

# Scheduler

In [8]:
def build_scheduler(optimizer, cfg):
    """
    - warmup_epochs 동안 lr를 0 → lr로 선형 증가
    - 이후 Cosine Annealing으로 min_lr까지 감소
    """
    lr            = cfg["optimizer"]["lr"]
    warmup_epochs = cfg["scheduler"]["warmup_epochs"]
    min_lr        = cfg["scheduler"]["min_lr"]
    num_epochs    = cfg["training"]["num_epochs"]

    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1e-6 / lr,
        end_factor=1.0,
        total_iters=warmup_epochs,
    )
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - warmup_epochs,
        eta_min=min_lr,
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[warmup_epochs],
    )

In [9]:
@torch.no_grad()
def evaluate(model, loader):
    """
    Returns:
        auc   : ROC-AUC (높을수록 좋음)
        lloss : Log-loss (낮을수록 좋음, DFDC 공식 지표)
        acc   : Accuracy
    """
    model.eval()
    all_logits, all_labels = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        all_logits.append(logits.cpu())
        all_labels.append(labels)

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    probs  = torch.sigmoid(logits).numpy()
    preds  = (probs >= 0.5).astype(int)

    auc   = roc_auc_score(labels, probs)
    lloss = log_loss(labels, probs)
    acc   = (preds == labels).mean()

    return {"auc": auc, "log_loss": lloss, "acc": acc}


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0

    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()


        # 훈련하다가 뜬 경고 메세지 없애기 !!  -> 이거랑 scaler도 함께 세트로 바꿔야 함.
        with torch.amp.autocast('cuda'):
        #with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)


        # Gradient clipping — ConvNeXt + Transformer 계열에 권장
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

# CSV

In [10]:
import re
import json
import tarfile
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm

DRIVE_DIR     = Path("/content/drive/MyDrive/Deepfake_Preprocessed")
LOCAL_TMP_DIR = Path("/content/meta_tmp")
CSV_SAVE_PATH = DRIVE_DIR / "dataset_metadata.csv"

In [11]:
from concurrent.futures import ThreadPoolExecutor

if CSV_SAVE_PATH.exists():
    print(f"이미 존재함, 스킵: {CSV_SAVE_PATH}")
else:

    tar_files = sorted(DRIVE_DIR.glob("dataset_chunk_*.tar"))
    print(f"발견된 tar: {len(tar_files)}개 -> meta.json 병렬 추출 시작")

    def extract_meta_from_tar(tar_path):
        local_records=[]
        with tarfile.open(tar_path) as tar:
            meta_members = [m for m in tar.getmembers()
                           if m.name.endswith("meta.json")]
            for member in meta_members:
                f = tar.extractfile(member)
                if f:
                    data = json.load(f)
                    local_records.append(data)
        return local_records

    records = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(tqdm(
            executor.map(extract_meta_from_tar, tar_files),
            total=len(tar_files),
            desc="병렬 추출 중"
        ))

    for r in results:
        records.extend(r)

    print(f"총 {len(records)}개 영상 메타데이터 수집 완료")

    # LOCAL_TMP_DIR.mkdir(parents=True, exist_ok=True)
    # tar_files = sorted(DRIVE_DIR.glob("dataset_chunk_*.tar"))
    # print(f"발견된 tar: {len(tar_files)}개 → meta.json 추출 시작")

    # records = []

    # for tar_path in tqdm(tar_files, desc="tar 순회 중"):
    #     with tarfile.open(tar_path) as tar:
    #         meta_members = [m for m in tar.getmembers() if m.name.endswith("meta.json")]
    #         for member in meta_members:
    #             tar.extract(member, path=LOCAL_TMP_DIR)

    # for meta_file in LOCAL_TMP_DIR.rglob("meta.json"):
    #     with open(meta_file, "r", encoding="utf-8") as f:
    #         records.append(json.load(f))

    # print(f"총 {len(records)}개 영상 메타데이터 수집 완료")

    def get_dfd_identity(path: Path):
        match = re.match(r"^(\d{2})(?:_\d{2})?__", path.stem)
        return int(match.group(1)) if match else None

    def get_split(video_path: str, source: str, meta_split:str= None) -> str:
        path = Path(video_path)

        if source == "CelebDF":
            return meta_split

        if source == "DFD":
            identity = get_dfd_identity(path)
            if identity is None:
                return "train"
            if identity <= 20:
                return "train"  # 1~22번 배우
            if identity <= 25:
                return "val"    # 23~25번 배우
            return "test"       # 26~28번 배우
        return "train" # CelebDF는 추후 추가

    def get_split_random(df: pd.DataFrame, seed: int=42) -> pd.DataFrame:
        from sklearn.model_selection import train_test_split

        train_idx, temp_idx = train_test_split(
            df.index, test_size=0.2, random_state=seed, stratify=df["label"]
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=seed, stratify=df.loc[temp_idx, "label"]
        )
        df["split"] = "train"
        df.loc[val_idx, "split"] = "val"
        df.loc[test_idx, "split"] = "test"
        return df

    df = pd.DataFrame(records)
    #df["split"] = df.apply(lambda row: get_split(row["video_path"], row["source"]), axis=1)
    #df = get_split_random(df)

    def assign_split(row):
        if row["source"] == "CelebDF":
            return row["split"]
        else:
            return get_split(row["video_path"], row["source"])
    df["split"] = df.apply(assign_split, axis=1)

    # ── 저장 ──
    df.to_csv(CSV_SAVE_PATH, index=False)
    print(f"\nCSV 저장 완료: {CSV_SAVE_PATH}")

    # 분할 결과 출력
    print("\n=== 분할 결과 ===")
    for split in ["train", "val", "test"]:
        sub = df[df["split"] == split]
        if sub.empty:
            continue
        print(f"  {split:5}: {len(sub):4}개 "
              f"(real={(sub.label==0).sum()}, fake={(sub.label==1).sum()})")

    shutil.rmtree(LOCAL_TMP_DIR)
    print("\n임시 폴더 정리 완료")

이미 존재함, 스킵: /content/drive/MyDrive/Deepfake_Preprocessed/dataset_metadata.csv


In [ ]:
import time
import tarfile
import shutil
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, log_loss
from torch.utils.data import WeightedRandomSampler
import subprocess
from concurrent.futures import ThreadPoolExecutor

def extract_single_tar(tar_path: Path, extract_root: str):
    """tar 파일 하나를 압축 해제"""
    subprocess.run(["tar", "-xf", str(tar_path), "-C", str(extract_root)], check=True)

def extract_chunks(tar_paths: list, extract_root: str="/content")-> None:
    """여러 tar 파일을 멀티스레딩으로 동시 압축 해제"""
    # 청크 개수(예: 8개)만큼 스레드를 생성해서 동시에 압축 풀기
    with ThreadPoolExecutor(max_workers=len(tar_paths)) as executor:
        for tar_path in tar_paths:
            executor.submit(extract_single_tar, tar_path, extract_root)

# def extract_chunks(tar_paths: list, extract_root: str = "/content") -> None:
#     """tar 파일 목록을 압축 해제"""
#     for tar_path in tar_paths:
#         with tarfile.open(tar_path) as tar:
#             tar.extractall(path=extract_root, filter='data')

def clear_local_crops(local_dir: Path) -> None:
    """로컬 processed_crops 폴더 초기화 (디스크 확보)"""
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)


def train(cfg):
    p   = cfg["path"]
    m   = cfg["model"]
    inp = cfg["input"]
    tr  = cfg["training"]
    opt = cfg["optimizer"]
    ckp = cfg["checkpoint"]

    local_dir    = Path(p["base_dir"])          # /content/processed_crops
    drive_dir    = Path(p["save_dir"]).parent   # /content/drive/MyDrive/Deepfake_Preprocessed
    CHUNK_SIZE   = 8                            # 한 번에 압축 해제할 tar 파일 수

    all_tar_files = sorted(drive_dir.glob("dataset_chunk_*.tar"))
    chunk_groups  = [all_tar_files[i:i+CHUNK_SIZE] for i in range(0, len(all_tar_files), CHUNK_SIZE)]
    print(f"전체 tar: {len(all_tar_files)}개 → {len(chunk_groups)}개 그룹 (그룹당 최대 {CHUNK_SIZE}개)")

    csv_path = drive_dir / "dataset_metadata.csv"
    df       = pd.read_csv(csv_path)
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df   = df[df["split"] == "val"].reset_index(drop=True)

    print(f"\nTrain: {len(train_df)}개 | Val: {len(val_df)}개")
    print(f"  Train — real: {(train_df.label==0).sum()}, fake: {(train_df.label==1).sum()}")
    print(f"  Val   — real: {(val_df.label==0).sum()},   fake: {(val_df.label==1).sum()}")

    df = pd.read_csv(csv_path)
    print(df.groupby(['split', 'label']).size())
    print(f"\n전체: {len(df)}개")
    print(f"real: {(df.label==0).sum()}개")
    print(f"fake: {(df.label==1).sum()}개")


    # ── 클래스 불균형 처리: pos_weight (전체 train 기준으로 한 번만 계산) ──
    n_real     = (train_df.label == 0).sum()
    n_fake     = (train_df.label == 1).sum()

    #pos_weight = torch.tensor([n_real / n_fake], device=device)   #  기존 pos_weight
    pos_weight = torch.tensor([1.0], device=device)                # 5.24 수정 pos_weight

    print(f"\npos_weight: {pos_weight.item():.3f} (real/fake 비율)")
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    global_class_counts = np.bincount(train_df["label"].values)
    global_class_weights = 1.0 / global_class_counts
    print(f"[Sampler Weights] Real(0): {global_class_weights[0]:.4f}, Fake(1): {global_class_weights[1]:.4f}")

    model = ConvNeXtArtifactDetector(
        model_name=m["model_name"],
        pretrained=True,
        dropout=m["dropout"],
        use_highpass=m["use_highpass"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=opt["lr"],
        weight_decay=opt["weight_decay"],
    )
    scheduler = build_scheduler(optimizer, cfg)

    scaler = torch.amp.GradScaler('cuda')

    save_dir = Path(p["save_dir"])
    save_dir.mkdir(parents=True, exist_ok=True)
    best_checkpoints = []

    # ── 체크포인트 이어서 학습하기 ──
    start_epoch = 1
    existing_ckpts = sorted(list(save_dir.glob("epoch*.pt")))

    if existing_ckpts:
        latest_ckpt_path = existing_ckpts[-1]
        print(f"기존 체크포인트가 있어, 이어서 학습하기: {latest_ckpt_path.name}")

        checkpoint = torch.load(latest_ckpt_path, map_location=device, weights_only=False)

        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])

        if "scheduler_state" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state"])
        if "scaler_state" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state"])

        start_epoch = checkpoint["epoch"] + 1

    else:
        print("기존 체크포인트 없음: 1번째 에포크부터 처음부터 학습을 시작합니다.")

    history = []
    print("\n학습 시작\n" + "="*60)

    for epoch in range(start_epoch, tr["num_epochs"] + 1):
        t0         = time.time()
        epoch_loss = 0.0
        n_chunks   = 0

        # Val Loss 계산을 위한 변수 초기화
        epoch_val_loss = 0.0
        val_total_samples = 0
        all_val_logits, all_val_labels = [], []

        for group_idx, tar_group in enumerate(chunk_groups):
            print(f"\n  [Epoch {epoch} | Group {group_idx+1}/{len(chunk_groups)}] "
                  f"압축 해제 중: {[t.name for t in tar_group]}")

            clear_local_crops(local_dir)
            extract_chunks(tar_group, extract_root=local_dir)

            extracted_ids = {p.parent.name for p in local_dir.glob("*/meta.json")}

            chunk_train_df = train_df[train_df["video_id"].isin(extracted_ids)].reset_index(drop=True)
            chunk_val_df   = val_df[val_df["video_id"].isin(extracted_ids)].reset_index(drop=True)

            # ── Train ──
            if not chunk_train_df.empty:
                train_ds = DeepfakeDataset(local_dir, chunk_train_df, train_transform, inp["max_frame"])
                labels = chunk_train_df["label"].values

                # class_counts = np.bincount(labels)
                # class_weights = 1.0 / class_counts
                # sample_weights = class_weights[labels]

                sample_weights = global_class_weights[labels]

                sampler = WeightedRandomSampler(
                    weights=torch.tensor(sample_weights, dtype=torch.float),
                    num_samples=len(sample_weights),
                    replacement=True
                )

                train_loader = DataLoader(
                    train_ds,
                    batch_size=tr["batch_size"],
                    #shuffle=True,
                    sampler=sampler,
                    num_workers=tr["num_workers"],
                    pin_memory=True,
                )
                chunk_loss  = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
                epoch_loss += chunk_loss
                n_chunks   += 1
                print(f"  └ train chunk_loss: {chunk_loss:.4f}")
            else:
                print(f"train 데이터 없음, 스킵")

            # ── Val ──
            if not chunk_val_df.empty:
                val_ds = DeepfakeDataset(local_dir, chunk_val_df, val_transform, inp["max_frame"])
                val_loader = DataLoader(
                    val_ds,
                    batch_size=tr["batch_size"],
                    shuffle=False,
                    num_workers=tr["num_workers"],
                    pin_memory=True,
                )
                model.eval()
                with torch.no_grad():
                    for imgs, labels in val_loader:
                        imgs = imgs.to(device)
                        labels = labels.to(device)

                        logits = model(imgs)

                        # 1차원으로 납작하게 맞춰서 BCE Loss 계산
                        loss = criterion(logits.view(-1), labels.float().view(-1))

                        # 데이터 개수(batch_size_current)를 곱해서 가중 평균용 손실합 계산
                        batch_size_current = imgs.size(0)
                        epoch_val_loss += loss.item() * batch_size_current
                        val_total_samples += batch_size_current

                        all_val_logits.append(logits.cpu())
                        all_val_labels.append(labels.cpu())
                model.train()
                print(f"  └ val chunk 수집: {len(chunk_val_df)}개")

        # ── Epoch 지표 산출 ──
        avg_loss = epoch_loss / max(n_chunks, 1)

        # 총 샘플 수로 나누어 완벽한 Val 가중 평균 계산
        avg_val_loss = epoch_val_loss / max(val_total_samples, 1)

        val_logits = torch.cat(all_val_logits)
        val_labels = torch.cat(all_val_labels).numpy()
        val_probs  = torch.sigmoid(val_logits).numpy()
        val_preds  = (val_probs >= 0.5).astype(int)

        val_metrics = {
            "auc":      roc_auc_score(val_labels, val_probs),
            "log_loss": log_loss(val_labels, val_probs),
            "acc":      (val_preds == val_labels).mean(),
            "loss":     round(avg_val_loss, 4),
        }

        # 스케줄러 업데이트 및 현재 LR 기록
        cur_lr  = optimizer.param_groups[0]["lr"]
        scheduler.step()

        elapsed = time.time() - t0

        log = {
            "epoch":       epoch,
            "train_loss":  round(avg_loss, 4),
            "val_loss":    round(avg_val_loss, 4),
            "val_auc":     round(val_metrics["auc"], 4),
            "val_logloss": round(val_metrics["log_loss"], 4),
            "val_acc":     round(val_metrics["acc"], 4),
            "lr":          round(cur_lr, 8),
        }
        history.append(log)

        print(
            f"\n[Epoch {epoch:02d}/{tr['num_epochs']}] "
            f"train_loss={log['train_loss']:.4f} | "
            f"val_loss={log['val_loss']:.4f} | "
            f"val_auc={log['val_auc']:.4f} | "
            f"val_logloss={log['val_logloss']:.4f} | "
            f"val_acc={log['val_acc']:.4f} | "
            f"lr={cur_lr:.2e} | "
            f"{elapsed:.1f}s"
        )

        ckpt_path = save_dir / f"epoch{epoch:02d}_auc{log['val_auc']:.4f}.pt"
        torch.save({
            "epoch":           epoch,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state":    scaler.state_dict(),
            "val_metrics":     val_metrics,
            "cfg":             cfg,
        }, ckpt_path)

        # 상위 K개 체크포인트만 남기고 삭제
        best_checkpoints.append((log["val_auc"], ckpt_path))
        best_checkpoints.sort(key=lambda x: x[0], reverse=True)

        while len(best_checkpoints) > ckp["save_top_k"]:
            _, old_path = best_checkpoints.pop()
            if old_path.exists():
                old_path.unlink()
                print(f"  └ 삭제: {old_path.name}")

    print("\n" + "="*60)
    print(f"학습 완료!")
    print(f"Best val_auc: {best_checkpoints[0][0]:.4f} → {best_checkpoints[0][1].name}")

    hist_df   = pd.DataFrame(history)
    hist_path = save_dir / "train_history.csv"
    hist_df.to_csv(hist_path, index=False)
    print(f"학습 기록 저장: {hist_path}")

    clear_local_crops(local_dir)

    return model, history


if __name__ == "__main__":
    model, history = train(CFG)

전체 tar: 34개 → 5개 그룹 (그룹당 최대 8개)

Train: 9271개 | Val: 455개
  Train — real: 1253, fake: 8018
  Val   — real: 64,   fake: 391
split  label
test   0          38
       1         303
train  0        1253
       1        8018
val    0          64
       1         391
dtype: int64

전체: 10067개
real: 1355개
fake: 8712개

pos_weight: 1.000 (real/fake 비율)
[Sampler Weights] Real(0): 0.0008, Fake(1): 0.0001
기존 체크포인트 없음: 1번째 에포크부터 처음부터 학습을 시작합니다.

학습 시작

  [Epoch 1 | Group 1/5] 압축 해제 중: ['dataset_chunk_000.tar', 'dataset_chunk_001.tar', 'dataset_chunk_002.tar', 'dataset_chunk_003.tar', 'dataset_chunk_004.tar', 'dataset_chunk_005.tar', 'dataset_chunk_006.tar', 'dataset_chunk_007.tar']
